In [1]:
import math
import torch
import torch.nn.functional as F

In [2]:
def scaled_dot_product_attention(query: torch.Tensor, key: torch.Tensor, value: torch.Tensor, mask: torch.Tensor = None) -> torch.Tensor:
    """
    Computes Scaled Dot-Product Attention.
    
    Args:
        query: Tensor of shape (batch_size, num_heads, seq_len_q, d_k)
        key: Tensor of shape (batch_size, num_heads, seq_len_k, d_k)
        value: Tensor of shape (batch_size, num_heads, seq_len_v, d_v)
        mask: Optional mask tensor of shape (batch_size, 1, seq_len_q, seq_len_k)
        
    Returns:
        output: Weighted context representations
        attention_weights: Softmax weights for visualization/analysis
    """
    d_k = query.shape[-1]
    
    # Step 1: Compute raw attention scores via matrix multiplication
    # Shape: (batch_size, num_heads, seq_len_q, seq_len_k)
    scores = torch.matmul(query, key.transpose(-2, -1)) / math.sqrt(d_k)
    
    # Step 2: Apply optional mask (e.g., for padding or causal decoding)
    if mask is not None:
        scores = scores.masked_fill(mask == 0, -1e9)
        
    # Step 3: Apply softmax to get normalized attention weights
    attention_weights = F.softmax(scores, dim=-1)
    
    # Step 4: Multiply by the Value matrix to get the final output
    output = torch.matmul(attention_weights, value)
    
    return output, attention_weights

# --- Example Usage ---
if __name__ == "__main__":
    batch_size = 2
    num_heads = 4
    seq_len = 5
    d_k = 16
    
    # Initialize random Q, K, V tensors simulating multi-head setup
    Q = torch.randn(batch_size, num_heads, seq_len, d_k)
    K = torch.randn(batch_size, num_heads, seq_len, d_k)
    V = torch.randn(batch_size, num_heads, seq_len, d_k)
    
    output, weights = scaled_dot_product_attention(Q, K, V)
    
    print("Output shape:", output.shape)
    print("Attention weights shape:", weights.shape)

Output shape: torch.Size([2, 4, 5, 16])
Attention weights shape: torch.Size([2, 4, 5, 5])


In [3]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model: int, num_heads: int):
        super(MultiHeadAttention, self).__init__()
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        
        assert self.d_k * num_heads == d_model, "d_model must be divisible by num_heads"
        
        # Linear layers for Q, K, V projections and final output projection
        self.q_linear = nn.Linear(d_model, d_model)
        self.k_linear = nn.Linear(d_model, d_model)
        self.v_linear = nn.Linear(d_model, d_model)
        self.out_linear = nn.Linear(d_model, d_model)
        
    def _split_heads(self, tensor: torch.Tensor, batch_size: int) -> torch.Tensor:
        """
        Reshape tensor to separate heads: (batch_size, seq_len, d_model) 
        -> (batch_size, num_heads, seq_len, d_k)
        """
        return tensor.view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)

    def forward(self, q: torch.Tensor, k: torch.Tensor, v: torch.Tensor, mask: torch.Tensor = None) -> torch.Tensor:
        batch_size = q.size(0)
        
        # 1. Linear projections
        q = self.q_linear(q)
        k = self.k_linear(k)
        v = self.v_linear(v)
        
        # 2. Split into multiple heads
        q = self._split_heads(q, batch_size)
        k = self._split_heads(k, batch_size)
        v = self._split_heads(v, batch_size)
        
        # 3. Apply scaled dot-product attention across all heads
        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.d_k)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, -1e9)
            
        attention_weights = F.softmax(scores, dim=-1)
        output = torch.matmul(attention_weights, v)
        
        # 4. Concatenate heads back and pass through output projection
        output = output.transpose(1, 2).contiguous().view(batch_size, -1, self.d_model)
        return self.out_linear(output)

# --- Example Usage ---
if __name__ == "__main__":
    d_model = 512
    num_heads = 8
    batch_size = 2
    seq_len = 10
    
    mha = MultiHeadAttention(d_model, num_heads)
    
    # Input tensor representing sequence embeddings
    x = torch.randn(batch_size, seq_len, d_model)
    
    out = mha(x, x, x)
    print("Multi-Head Attention Output Shape:", out.shape)

Multi-Head Attention Output Shape: torch.Size([2, 10, 512])
